## Setup — run this first

Mounts Drive, points the notebook at your project folder, and installs
what's missing. No git, no tokens.

**Your Drive folder must look like this:**

```
MyDrive/Ghana_Dropout_Project_R02/
├── config.py          <- these three at the TOP level,
├── losses.py             not inside notebooks/
├── pipeline.py
├── requirements.txt
├── notebooks/         <- the 11 notebooks
└── data-raw/
    └── ghana_dropout_study_M.xlsx
```

`results/`, `figures/`, `models/` and `data-processed/` are created for you.

Drive saves as it goes, so there is nothing to push — but see the checklist
in the last cell before you submit.


In [1]:
# ============================================================
# SETUP — Google Drive. Run first. Safe to re-run.
# ============================================================
import os, sys, subprocess
from pathlib import Path

PROJECT = "/content/drive/MyDrive/Ghana_Dropout_Project_R02"   # <-- edit if yours differs
RAW_XLSX_NAME = "ghana_dropout_study_M.xlsx"

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")

    root = Path(PROJECT)
    if not root.exists():
        raise FileNotFoundError(
            f"{PROJECT} does not exist.\n"
            "Create that folder in My Drive and put config.py, losses.py, "
            "pipeline.py, requirements.txt, the notebooks/ folder and "
            "data-raw/ inside it."
        )

    # the three modules must sit at the project root, not in notebooks/
    missing = [m for m in ("config.py", "losses.py", "pipeline.py")
               if not (root / m).exists()]
    if missing:
        stray = [m for m in missing if (root / "notebooks" / m).exists()]
        msg = f"Missing from {PROJECT}: {missing}"
        if stray:
            msg += (f"\n{stray} are in notebooks/ instead. Move them UP one "
                    "level, into the project folder itself. If they stay in "
                    "notebooks/, that folder gets treated as the project root "
                    "and results/ is written in the wrong place.")
        raise FileNotFoundError(msg)

    os.chdir(root)
    os.environ["DROPOUT_REPO"] = str(root)

    # Forget any previously loaded copy of the project modules. Python keeps
    # the first version it imported for the whole session, so an edited
    # config.py is silently ignored until the runtime restarts. This makes
    # every run use the files currently in Drive.
    for _m in ("config", "losses", "pipeline"):
        sys.modules.pop(_m, None)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

    # ---- dependencies: only install what is actually missing ------------
    need = []
    for mod, pkg in [("lightgbm", "lightgbm"), ("shap", "shap"),
                     ("catboost", "catboost"), ("xgboost", "xgboost"),
                     ("imblearn", "imbalanced-learn"), ("openpyxl", "openpyxl")]:
        try:
            __import__(mod)
        except ImportError:
            need.append(pkg)
    if need:
        print("installing:", need)
        subprocess.run(f"pip install -q {' '.join(need)}", shell=True)
    else:
        print("all dependencies present")

    # ---- raw workbook ---------------------------------------------------
    (root / "data-raw").mkdir(exist_ok=True)
    xlsx = root / "data-raw" / RAW_XLSX_NAME
    if xlsx.exists():
        print(f"raw workbook: {xlsx.name}")
    else:
        loose = list(root.glob(RAW_XLSX_NAME)) + list(root.glob(f"**/{RAW_XLSX_NAME}"))
        if loose:
            import shutil
            shutil.copy(loose[0], xlsx)
            print(f"copied {loose[0]} -> data-raw/")
        else:
            print(f"NOT FOUND: data-raw/{RAW_XLSX_NAME}\n"
                  "Notebook 1 needs it. Notebooks 2-9 read "
                  "data-processed/cleaned_data.csv instead and are fine "
                  "without it.")

    print(f"\nPROJECT : {os.getcwd()}")
else:
    print("Not in Colab — paths resolve from the project root.")


Mounted at /content/drive
installing: ['catboost']
raw workbook: ghana_dropout_study_M.xlsx

PROJECT : /content/drive/MyDrive/Ghana_Dropout_Project_R02


# Notebook 7 — Explainability (signed SHAP)

## What changed from R01

| Change | Reason |
|---|---|
| **Signed** SHAP values retained | Q14: R01 reported mean *absolute* SHAP only, so the submission could not say whether higher family income predicts more dropout or less. Direction is the finding in a dropout model, and R01 discarded it before reporting |
| **Family-level attribution across all features** | Q13: R01 reported a truncated top five plus two named ranks. The full ranking was committed all along, so the truncation had no data cost to reverse |
| Explained on a **CV-held-out fold**, not the frozen test partition | GATE-1(iii). R01 explained on the test partition, which is the right instinct against memorisation but was one of the six notebooks scoring it |
| Both SHAP artefacts **labelled by model** | Q14: `shap_feature_importance.csv` (38 features) and `feature_importance_focal.csv` (41) disagreed by a factor of two on the top feature with no explanation |
| Minority-class attribution uncertainty reported | Q14: attribution rests on ~18 positive instances and R01 stated no uncertainty |
| Imports `losses.py` | R01 redefined the focal loss locally "so this notebook has no dependency on losses.py" — which is precisely how the two copies drifted apart |
| Composite verification is reported *with* interpretation | M17 said the ranking would be "reported in R6 without interpretation". The ablation already showed the composites cost performance, so the verification has an answer and it should be stated |

In [2]:
# ---- bootstrap: repo-relative imports, no drive.mount, no hard-coded path ----
import sys, os
from pathlib import Path

def _find_repo(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p, *p.parents]:
        if (c / "config.py").exists():
            return c
    return p

REPO = Path(os.environ["DROPOUT_REPO"]) if os.environ.get("DROPOUT_REPO") else _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# In Colab, clone the repo first, then run:
#     import os; os.environ["DROPOUT_REPO"] = "/content/student-dropout-prediction-ghana"
# Raw pupil-level data is NOT in the repo (ethics); place it under data-raw/
# locally. Nothing below calls drive.mount().

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
from config import *
from pipeline import (preprocess_inside_fold, frozen_split, cv_splits,
                      fit_arm, ARMS, ARM_LABELS, HEADLINE, OLD_HEADLINE,
                      raw_feature_cols)

banner("NOTEBOOK 7 — SIGNED SHAP")
OUT = run_dir("notebook07_shap")
capture_environment(OUT)

try:
    import shap
except ImportError:
    raise ImportError("pip install shap")

df = pd.read_csv(CLEANED_CSV)
train_pool, test_holdout = frozen_split(df)

# Explain on a CV-held-out fold of the training pool. The frozen test
# partition stays untouched until Notebook 8.
tr, vl = cv_splits(train_pool, SPLIT_SEED, n_repeats=1)[0]
X_tr, y_tr, X_vl, y_vl, meta = preprocess_inside_fold(
    train_pool.iloc[tr], train_pool.iloc[vl])
N_POS_EXPLAINED = int((y_vl == 1).sum())
print(f"explaining on {len(y_vl)} held-out rows, {N_POS_EXPLAINED} of them dropout")
print(f"{meta['n_features']} features")
print("\nThe test partition is NOT used here.")

NOTEBOOK 7 — SIGNED SHAP
repo            : /content/drive/MyDrive/Ghana_Dropout_Project_R02
provenance      : NONE — set FREEZE_TAG in config.py before scoring the test set
school_handling : drop
FEATURE SET     : records_plus_questionnaire   (SECONDARY — includes friend-reported questionnaire items)
primary metric  : auc_pr
explaining on 157 held-out rows, 14 of them dropout
47 features

The test partition is NOT used here.


In [4]:
# ---- signed SHAP per arm, each artefact labelled ----------------------
def shap_for_arm(arm_name):
    model, predict, cols = fit_arm(arm_name, X_tr, y_tr, SPLIT_SEED)
    sv = shap.TreeExplainer(model).shap_values(X_vl[cols])
    if isinstance(sv, list):
        sv = sv[1]
    sv = np.asarray(sv)
    if sv.ndim == 3:
        sv = sv[:, :, 1]

    pos = (y_vl.to_numpy() == 1)
    t = pd.DataFrame({
        "feature": cols,
        "mean_abs_shap": np.abs(sv).mean(axis=0),
        "mean_signed_shap": sv.mean(axis=0),
        "sd_shap": sv.std(axis=0),
        "mean_abs_shap_positives": (np.abs(sv[pos]).mean(axis=0)
                                    if pos.sum() else np.nan),
        "mean_signed_shap_positives": (sv[pos].mean(axis=0)
                                       if pos.sum() else np.nan),
        "se_signed_positives": (sv[pos].std(axis=0, ddof=1) / np.sqrt(pos.sum())
                                if pos.sum() > 1 else np.nan),
    })
    # DIRECTION: does a HIGHER value of the feature push the prediction up or
    # down? Measured as the rank correlation between each feature's values and
    # its SHAP values. The MEAN signed SHAP is not a direction — positive and
    # negative contributions cancel across pupils and it lands near zero for
    # almost every feature, which produced nonsense like "higher English
    # scores raise dropout risk" in the first run.
    from scipy.stats import spearmanr
    corrs = []
    for k, c in enumerate(cols):
        x = pd.to_numeric(X_vl[c], errors="coerce").to_numpy(dtype=float)
        if np.nanstd(x) == 0 or np.std(sv[:, k]) == 0:
            corrs.append(0.0); continue
        r = spearmanr(x, sv[:, k], nan_policy="omit").correlation
        corrs.append(0.0 if np.isnan(r) else float(r))
    t["value_shap_corr"] = corrs
    t["direction"] = np.where(
        t["feature"].isin(NOMINAL_COLS), "categorical - see beeswarm",
        np.where(t["value_shap_corr"] > 0.2, "higher value -> MORE dropout risk",
        np.where(t["value_shap_corr"] < -0.2, "higher value -> LESS dropout risk",
                 "no consistent direction")))
    t["family"] = t["feature"].map(family_of)
    t["is_composite"] = t["feature"].isin(COMPOSITES)
    # MODEL LABEL — the fix for the two disagreeing R01 artefacts
    t["model_arm"] = arm_name
    t["model_label"] = ARM_LABELS[arm_name]
    t["n_features_in_model"] = len(cols)
    t["n_positive_instances_explained"] = N_POS_EXPLAINED
    t = t.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    t["rank"] = t.index + 1
    return t, sv, cols

tables, raw_sv = {}, {}
for arm in [HEADLINE[1], HEADLINE[0], OLD_HEADLINE[0]]:
    t, sv, cols = shap_for_arm(arm)
    tables[arm] = t
    raw_sv[arm] = (sv, cols)
    fn = f"shap_signed__{arm}.csv"
    t.to_csv(OUT / fn, index=False)
    print(f"\n=== {arm} — {ARM_LABELS[arm]} ===")
    print(f"    {len(cols)} features, file: {fn}")
    print(t.head(12)[["rank", "feature", "family", "mean_abs_shap",
                      "value_shap_corr", "direction"]].round(3).to_string(index=False))


=== G_all_focal_noW — E-LightGBM (composites, focal loss) ===
    47 features, file: shap_signed__G_all_focal_noW.csv
 rank                         feature                 family  mean_abs_shap  value_shap_corr                         direction
    1              average_exam_score   academic_performance          0.547            0.132           no consistent direction
    2           attendance_risk_index  attendance_trajectory          0.308            0.319 higher value -> MORE dropout risk
    3             family_income_level     household_economic          0.268           -0.373 higher value -> LESS dropout risk
    4               term_2_attendance  attendance_trajectory          0.236           -0.399 higher value -> LESS dropout risk
    5               term_1_attendance  attendance_trajectory          0.187           -0.431 higher value -> LESS dropout risk
    6    parent_attends_school_events behavioural_engagement          0.181           -0.239 higher value -> LESS dropo

In [5]:
# ---- reconcile the two R01 artefacts (Q14) ---------------------------
a, b = HEADLINE[1], OLD_HEADLINE[0]
rec = (tables[a][["feature", "rank", "mean_abs_shap"]]
       .merge(tables[b][["feature", "rank", "mean_abs_shap"]],
              on="feature", how="outer", suffixes=(f"__{a}", f"__{b}")))
rec["abs_shap_ratio"] = (rec[f"mean_abs_shap__{a}"] / rec[f"mean_abs_shap__{b}"])
rec = rec.sort_values(f"mean_abs_shap__{a}", ascending=False)
rec.to_csv(OUT / "shap_artefact_reconciliation.csv", index=False)
print("SHAP ARTEFACT RECONCILIATION\n")
print(rec.head(12).round(4).to_string(index=False))
print(f"""
R01 committed two SHAP files that disagreed by a factor of two on the top
feature, with nothing in the manuscript saying which model produced which.
They are coherent as the baseline ({b}, {tables[b]['n_features_in_model'].iloc[0]} features) and the
proposed model ({a}, {tables[a]['n_features_in_model'].iloc[0]} features). Every file this notebook writes
carries model_arm and model_label columns, and the filename names the arm.
Attribution magnitudes are NOT comparable across models with different
feature counts — say that wherever both are cited.""")

SHAP ARTEFACT RECONCILIATION

                        feature  rank__G_all_focal_noW  mean_abs_shap__G_all_focal_noW  rank__LEGACY_A  mean_abs_shap__LEGACY_A  abs_shap_ratio
             average_exam_score                      1                          0.5473             1.0                   1.6900          0.3238
          attendance_risk_index                      2                          0.3084             NaN                      NaN             NaN
            family_income_level                      3                          0.2679             5.0                   0.7334          0.3653
              term_2_attendance                      4                          0.2361             2.0                   1.2344          0.1913
              term_1_attendance                      5                          0.1867             4.0                   0.9815          0.1902
   parent_attends_school_events                      6                          0.1812             6.0    

In [6]:
# ---- family-level attribution (Q13) ---------------------------------
arm = HEADLINE[1]
t = tables[arm]
fam = (t.groupby("family")
       .agg(total_abs_shap=("mean_abs_shap", "sum"),
            mean_abs_shap=("mean_abs_shap", "mean"),
            mean_signed=("mean_signed_shap", "mean"),
            n_features=("feature", "size"),
            top_feature=("feature", "first"))
       .sort_values("total_abs_shap", ascending=False).reset_index())
fam["share_pct"] = (100 * fam["total_abs_shap"] / fam["total_abs_shap"].sum()).round(1)
fam["model_arm"] = arm
fam.to_csv(OUT / f"shap_family_attribution__{arm}.csv", index=False)

print("FAMILY-LEVEL ATTRIBUTION — replaces the top-five list in R6")
print(f"(all {len(t)} features, {arm})\n")
print(fam.round(4).to_string(index=False))

unclass = t[t["family"] == "other_unclassified"]
if len(unclass):
    print(f"\n{len(unclass)} features unclassified — add keywords to "
          f"config.FEATURE_FAMILIES:")
    print("   ", ", ".join(unclass["feature"].head(15)))

if SCHOOL_COL in set(t["feature"]):
    r = t[t["feature"] == SCHOOL_COL].iloc[0]
    print(f"\n*** {SCHOOL_COL} ranks {int(r['rank'])} of {len(t)} "
          f"(mean |SHAP| {r['mean_abs_shap']:.4f}) ***")
    print("An administrative identifier is not a construct — it is a cluster "
          "label, and its presence means the model has learned which school a "
          "record came from. Report it SEPARATELY from the construct families "
          "and say what it means. Setting config.SCHOOL_HANDLING='drop' "
          "removes it; that is decision D1.")
else:
    print(f"\n{SCHOOL_COL} excluded from the feature matrix "
          f"(SCHOOL_HANDLING={SCHOOL_HANDLING!r}), so no school-administrative "
          "family appears. State this in M9 and R6.")

plt.figure(figsize=(8, 4))
o = fam.sort_values("total_abs_shap")
plt.barh(o["family"], o["total_abs_shap"], color="steelblue")
plt.xlabel("summed mean |SHAP|"); plt.title(f"Family attribution — {arm}")
plt.tight_layout(); plt.savefig(OUT / "figures/family_attribution.png", dpi=200)
plt.close()

FAMILY-LEVEL ATTRIBUTION — replaces the top-five list in R6
(all 47 features, G_all_focal_noW)

                family  total_abs_shap  mean_abs_shap  mean_signed  n_features                   top_feature  share_pct       model_arm
  academic_performance          0.8831         0.1104      -0.0048           8            average_exam_score       28.7 G_all_focal_noW
 attendance_trajectory          0.7609         0.1268       0.0172           6         attendance_risk_index       24.7 G_all_focal_noW
behavioural_engagement          0.6970         0.0697      -0.0023          10  parent_attends_school_events       22.6 G_all_focal_noW
    household_economic          0.5041         0.0297       0.0012          17           family_income_level       16.4 G_all_focal_noW
      safety_wellbeing          0.1219         0.0610       0.0005           2                safety_at_home        4.0 G_all_focal_noW
           demographic          0.0676         0.0338       0.0036           2 age_at_st

In [7]:
# ---- DIRECTION, and the theory test (Q13, Q14) -----------------------
print("SIGNED DIRECTION OF THE TOP 15 FEATURES")
print("This is the table R01 could not produce, and it is what a domain")
print("examiner asks about first: 'which way does it point?'\n")
top = t.head(15)[["rank", "feature", "family", "mean_abs_shap",
                  "value_shap_corr", "direction"]]
print(top.round(4).to_string(index=False))

drift = top.copy()
drift["expected_direction"] = ""      # fill from your adopted theory
drift["matches_expectation"] = ""     # yes / no
drift["diagnosis"] = ""               # operationalisation / data / theory misfit
drift.to_csv(OUT / "construct_drift_worksheet.csv", index=False)
print("\nconstruct_drift_worksheet.csv written — fill the three blank columns. "
      "Q14 asks you to classify each mismatch as an operationalisation "
      "failure, a data failure, or a theory misfit.")

# The candidate drift the examination identified, now checkable
sv_col = "socioeconomic_vulnerability_score"
inc_col = SOCIOECONOMIC_COLS["family_income"]
for c in (sv_col, inc_col):
    if c in set(t["feature"]):
        r = t[t["feature"] == c].iloc[0]
        print(f"\n{c}: rank {int(r['rank'])}/{len(t)}, "
              f"mean |SHAP| {r['mean_abs_shap']:.4f}, {r['direction']}")
if sv_col in set(t["feature"]) and inc_col in set(t["feature"]):
    rs = int(t[t['feature']==sv_col]['rank'].iloc[0])
    ri = int(t[t['feature']==inc_col]['rank'].iloc[0])
    print(f"""
R01 pattern: the engineered composite ranked 24th of 41 (0.017) while its raw
ingredient family_income_level ranked 4th (0.225). Engineered construct dead,
raw ingredient alive — the signature of a broken operationalisation, not of a
theory being wrong about the world. The cause was the alphabetical encoding
(see Notebook 3). Current ranks: composite {rs}, raw ingredient {ri}.
If the composite has climbed, the fault was the encoding. If it is still
inert, the composite's construction is wrong and that is worth reporting.""")

SIGNED DIRECTION OF THE TOP 15 FEATURES
This is the table R01 could not produce, and it is what a domain
examiner asks about first: 'which way does it point?'

 rank                         feature                 family  mean_abs_shap  value_shap_corr                         direction
    1              average_exam_score   academic_performance         0.5473           0.1318           no consistent direction
    2           attendance_risk_index  attendance_trajectory         0.3084           0.3192 higher value -> MORE dropout risk
    3             family_income_level     household_economic         0.2679          -0.3729 higher value -> LESS dropout risk
    4               term_2_attendance  attendance_trajectory         0.2361          -0.3987 higher value -> LESS dropout risk
    5               term_1_attendance  attendance_trajectory         0.1867          -0.4307 higher value -> LESS dropout risk
    6    parent_attends_school_events behavioural_engagement         0.1812   

In [8]:
# ---- beeswarm and composite verification (M17) -----------------------
sv, cols = raw_sv[arm]
plt.figure()
shap.summary_plot(sv, X_vl[cols], max_display=20, show=False)
plt.title(f"Signed SHAP — {ARM_LABELS[arm]} (raw log-odds)")
plt.tight_layout()
plt.savefig(OUT / "figures/shap_beeswarm.png", dpi=300, bbox_inches="tight")
plt.close()

plt.figure()
shap.summary_plot(sv, X_vl[cols], plot_type="bar", max_display=20, show=False)
plt.title(f"mean |SHAP| — {ARM_LABELS[arm]}")
plt.tight_layout()
plt.savefig(OUT / "figures/shap_bar.png", dpi=300, bbox_inches="tight")
plt.close()
print("figures ->", OUT / "figures")

print("\nCOMPOSITE VERIFICATION (M17's stated purpose)")
comp = t[t["is_composite"]][["rank", "feature", "mean_abs_shap",
                             "mean_signed_shap", "direction"]]
print(comp.round(4).to_string(index=False))
n_raw = len(raw_feature_cols(X_vl))
print(f"\n{len(comp)} composites among {len(t)} features "
      f"({n_raw} raw). Composite ranks: {sorted(comp['rank'].tolist())}")
print("""
M17 said this ranking would be "reported in R6 without interpretation". It
has an interpretation and withholding it does not help: the ablation already
shows the composites reduced AUC-PR. A composite that ranks high in SHAP
while the ablation says it costs performance means the model uses it and is
worse for doing so -- which is redundancy with the raw ingredients, not
signal. Say that, and cite both tables.""")

write_manifest(OUT, {"notebook": "07_shap", "test_set_scored": False,
                     "explained_on": "CV-held-out fold of the training pool",
                     "n_instances_explained": int(len(y_vl)),
                     "n_positive_explained": N_POS_EXPLAINED,
                     "arms_explained": list(tables),
                     "signed_values_retained": True})

figures -> /content/drive/MyDrive/Ghana_Dropout_Project_R02/results/notebook07_shap/20260922T231223Z_records_plus_questionnaire/figures

COMPOSITE VERIFICATION (M17's stated purpose)
 rank                           feature  mean_abs_shap  mean_signed_shap                         direction
    2             attendance_risk_index         0.3084            0.0504 higher value -> MORE dropout risk
    7       behavioral_engagement_index         0.1712           -0.0276 higher value -> LESS dropout risk
   24 socioeconomic_vulnerability_score         0.0163            0.0006 higher value -> MORE dropout risk

3 composites among 47 features (44 raw). Composite ranks: [2, 7, 24]

M17 said this ranking would be "reported in R6 without interpretation". It
has an interpretation and withholding it does not help: the ablation already
shows the composites reduced AUC-PR. A composite that ranks high in SHAP
while the ablation says it costs performance means the model uses it and is
worse for doing s

{'run_dir': 'results/notebook07_shap/20260922T231223Z_records_plus_questionnaire',
 'generated_utc': '2026-09-22T23:12:44Z',
 'git': {'commit': '',
  'branch': '',
  'dirty': False,
  'dirty_paths': [],
  'no_git': True,
  'freeze_tag': ''},
 'host': '646b278326f7',
 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.39',
 'python': '3.13.15',
 'protocol': {'split_seed': 42,
  'test_size': 0.2,
  'seeds': [42, 123, 456, 789, 1024, 2048, 3333, 5555, 7777, 9999],
  'n_splits': 5,
  'n_repeats': 5,
  'primary_metric': 'auc_pr',
  'shared_params': {'n_estimators': 300,
   'num_leaves': 31,
   'learning_rate': 0.05,
   'subsample': 0.8,
   'colsample_bytree': 0.8,
   'verbosity': -1},
  'gamma_reported': 2.0,
  'alpha_reported': 0.75,
  'school_handling': 'drop',
  'feature_set': 'records_plus_questionnaire',
  'active_composites': ['attendance_risk_index',
   'socioeconomic_vulnerability_score',
   'behavioral_engagement_index'],
  'fairness_threshold': 0.1,
  'drop_derived_duplicates': True,


---

## Before you submit

Drive has already saved everything — nothing to push. But two things still
have to happen before submission, and neither is automatic.


In [9]:
# ---- what this run produced, and what is still owed ----
import os, sys
from pathlib import Path

try:
    latest = sorted(Path(OUT).parent.glob("*"))[-1]
    files = sorted(p.relative_to(OUT).as_posix() for p in Path(OUT).rglob("*")
                   if p.is_file())
    print(f"run directory : {Path(OUT).relative_to(REPO)}")
    print(f"files written : {len(files)}")
    for f in files:
        print("   ", f)
except Exception as e:
    print("no run directory recorded in this session:", e)

print("""
────────────────────────────────────────────────────────────────────
STILL OWED BEFORE SUBMISSION — neither happens by itself

1. UPLOAD THE PROJECT TO GITHUB, ONCE.
   Q4 failed because the repository was not runnable from a clone. Drive
   is fine for working; the repo is the deliverable. When the analysis is
   finished, drag the whole project folder into GitHub in one upload —
   EXCEPT data-raw/ and anything holding pupil rows. The notebooks resolve
   paths relative to the project root, so they run from a clone unchanged.

   Do NOT upload:  data-raw/, data-processed/cleaned_data.csv,
                   any *_snapshot.csv
   DO upload:      config.py, losses.py, pipeline.py, notebooks/,
                   requirements.txt, README.md, and all of results/

2. SET config.FREEZE_TAG BEFORE SCORING THE TEST SET.
   Without git there is no commit hash to anchor the freeze to. Put a
   fixed dated string in config.py — e.g. "R02-freeze-2026-09-25-1430" —
   at the moment you freeze the configuration, and never revise it.
   Notebook 8 refuses to score the test set until it is set.
────────────────────────────────────────────────────────────────────""")


run directory : results/notebook07_shap/20260922T231223Z_records_plus_questionnaire
files written : 12
    RUN_MANIFEST.json
    construct_drift_worksheet.csv
    environment_versions.csv
    figures/family_attribution.png
    figures/shap_bar.png
    figures/shap_beeswarm.png
    pip_freeze.txt
    shap_artefact_reconciliation.csv
    shap_family_attribution__G_all_focal_noW.csv
    shap_signed__E_all_ce_noW.csv
    shap_signed__G_all_focal_noW.csv
    shap_signed__LEGACY_A.csv

────────────────────────────────────────────────────────────────────
STILL OWED BEFORE SUBMISSION — neither happens by itself

1. UPLOAD THE PROJECT TO GITHUB, ONCE.
   Q4 failed because the repository was not runnable from a clone. Drive
   is fine for working; the repo is the deliverable. When the analysis is
   finished, drag the whole project folder into GitHub in one upload —
   EXCEPT data-raw/ and anything holding pupil rows. The notebooks resolve
   paths relative to the project root, so they run from 